# Delhi Climate Time Series (Azure ML): Track, Register, Deploy, Invoke

Dataset files (same schema):
- `DailyDelhiClimateTrain.csv`
- `DailyDelhiClimateTest.csv`

Goal: predict `meantemp` using previous-day (lag-1) features.

Reference (deployment tutorial): https://learn.microsoft.com/azure/machine-learning/tutorial-deploy-model?view=azureml-api-2


In [ ]:
# Install everything needed to run this notebook on a fresh Azure ML Compute Instance.
# Run once, then restart the kernel.

%pip install -U azure-ai-ml azure-identity azureml-fsspec "mlflow<3" azureml-mlflow pandas scikit-learn


In [ ]:
import datetime
import json

import pandas as pd
import mlflow
import mlflow.sklearn

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


## Connect To Workspace + Configure MLflow
Uses `config.json` (Studio notebook).


In [ ]:
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

ws = ml_client.workspaces.get(name=ml_client.workspace_name)
mlflow.set_tracking_uri(ws.mlflow_tracking_uri)
mlflow.set_experiment("delhi-climate-timeseries")

print("Workspace:", ml_client.workspace_name)
print("Location:", ws.location)
print("MLflow tracking URI:", mlflow.get_tracking_uri())


## Load Train/Test From Azure ML Data Asset

Upload the folder `TimeSeries_Project/` as a **folder data asset** in Azure ML.
Then set the name/version below to match your asset (example format: `azureml:<name>:<version>`).

This notebook expects these two files inside the asset folder:
- `DailyDelhiClimateTrain.csv`
- `DailyDelhiClimateTest.csv`


In [ ]:
DATA_ASSET_NAME = "delhi_climate_dataset"
DATA_ASSET_VERSION = "1"

asset = ml_client.data.get(name=DATA_ASSET_NAME, version=DATA_ASSET_VERSION)
base = str(asset.path).rstrip("/")

train_uri = base + "/DailyDelhiClimateTrain.csv"
test_uri  = base + "/DailyDelhiClimateTest.csv"

train_df = pd.read_csv(train_uri)
test_df  = pd.read_csv(test_uri)

train_df.head()


## Feature Engineering (Lag-1)

We learn a supervised model to predict `meantemp[t]` from previous day values:
- meantemp_lag1
- humidity_lag1
- wind_speed_lag1
- meanpressure_lag1


In [ ]:
def make_lag1_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    for col in ["meantemp", "humidity", "wind_speed", "meanpressure"]:
        df[f"{col}_lag1"] = df[col].shift(1)

    # Predict today's meantemp
    df = df.dropna().reset_index(drop=True)
    return df

full = pd.concat([train_df, test_df], ignore_index=True)
full_feat = make_lag1_features(full)

test_start = pd.to_datetime(test_df["date"]).min()
train_feat = full_feat[full_feat["date"] < test_start].copy()
test_feat  = full_feat[full_feat["date"] >= test_start].copy()

feature_cols = ["meantemp_lag1", "humidity_lag1", "wind_speed_lag1", "meanpressure_lag1"]
target_col = "meantemp"

X_train = train_feat[feature_cols]
y_train = train_feat[target_col]
X_test  = test_feat[feature_cols]
y_test  = test_feat[target_col]

X_train.shape, X_test.shape


## Train + Track With MLflow

Model: `StandardScaler` + `Ridge` regression.


In [ ]:
alpha = 1.0
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=alpha)),
])

with mlflow.start_run(run_name="ridge_lag1") as run:
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("target", target_col)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Compute RMSE without relying on mean_squared_error(..., squared=False) for compatibility
    rmse = mean_squared_error(y_test, preds) ** 0.5
    mae = mean_absolute_error(y_test, preds)
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("mae", float(mae))

    input_example = pd.DataFrame([X_test.iloc[0].to_dict()])
    mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example)

    run_id = run.info.run_id

print("Run ID:", run_id)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")


## Register Model In Azure ML


In [ ]:
local_model_dir = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="model")

model_name = "delhi-climate-forecast-mlflow"
registered_model = ml_client.models.create_or_update(
    Model(
        name=model_name,
        type=AssetTypes.MLFLOW_MODEL,
        path=local_model_dir,
        description="Delhi climate meantemp forecast (lag-1 Ridge, MLflow)",
    )
)

print("Registered model:", registered_model.name, registered_model.version)


## Deploy (Managed Online Endpoint)

If deployment fails with `SubscriptionNotRegistered`, register these providers in your subscription (Portal -> Subscription -> Resource providers):
- `Microsoft.MachineLearningServices`
- `Microsoft.ContainerRegistry`
- `Microsoft.Storage`
- `Microsoft.KeyVault`
- `Microsoft.Network`
- `Microsoft.ManagedIdentity`


In [ ]:
endpoint_name = "delhi-climate-" + datetime.datetime.now().strftime("%m%d%H%M%S")

try:
    endpoint = ManagedOnlineEndpoint(
        name=endpoint_name,
        auth_mode="key",
        location=ws.location,
    )
    endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

    deployment = ManagedOnlineDeployment(
        name="blue",
        endpoint_name=endpoint_name,
        model=registered_model,
        instance_type="Standard_D2ds_v5",
        instance_count=1,
    )
    ml_client.online_deployments.begin_create_or_update(deployment).result()

    endpoint = ml_client.online_endpoints.get(name=endpoint_name)
    endpoint.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()

    endpoint = ml_client.online_endpoints.get(name=endpoint_name)
    print("Endpoint name:", endpoint.name)
    print("Provisioning state:", endpoint.provisioning_state)
    print("Scoring URI:", endpoint.scoring_uri)

except HttpResponseError as e:
    print("Deploy failed:")
    print(str(e))

    resp = getattr(e, "response", None)
    if resp is not None:
        try:
            print(json.dumps(resp.json(), indent=2))
        except Exception:
            pass
    raise


## Invoke Endpoint (Prediction)

The deployed model expects the lag feature columns (same as `feature_cols`).


In [ ]:
row = X_test.iloc[0]
sample_request = {
    "input_data": {
        "columns": feature_cols,
        "data": [[float(row[c]) for c in feature_cols]],
    }
}

request_file = "sample-request-delhi.json"
with open(request_file, "w") as f:
    json.dump(sample_request, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name="blue",
    request_file=request_file,
)
print(response)
